In [12]:
# OASIS INFOBYTE - DATA ANALYTICS INTERNSHIP
# TASK 3: DATA CLEANING - TITANIC DATASET

import pandas as pd
import numpy as np
from IPython.display import display



1. Record the original dataset information


In [13]:
original_rows = 891
original_columns = 12

print("=" * 60)
print("OASIS INFOBYTE - TASK 3: DATA CLEANING")
print("=" * 60)

print("\nCURRENT DATASET SHAPE:", df.shape)

# Save a snapshot of the current dataset for comparison
df_clean = df.copy()



OASIS INFOBYTE - TASK 3: DATA CLEANING

CURRENT DATASET SHAPE: (891, 12)


2.Handle Missing Values

In [14]:
# Fill Age using median if missing values remain
if df_clean['Age'].isnull().any():
    df_clean['Age'] = df_clean['Age'].fillna(
        df_clean['Age'].median()
    )

# Fill Embarked using mode if missing values remain
if df_clean['Embarked'].isnull().any():
    df_clean['Embarked'] = df_clean['Embarked'].fillna(
        df_clean['Embarked'].mode()[0]
    )

print("\n1. MISSING VALUES HANDLED")



1. MISSING VALUES HANDLED


3.Investigate Cabin

In [15]:
cabin_missing = df_clean['Cabin'].isnull().sum()
cabin_missing_pct = df_clean['Cabin'].isnull().mean() * 100

print("\n2. CABIN INVESTIGATION")
print("Missing Cabin values:", cabin_missing)
print("Missing Cabin percentage:", round(cabin_missing_pct, 2), "%")

# Retain Cabin because most values are missing.
# Create a feature indicating whether Cabin information exists.
df_clean['CabinKnown'] = df_clean['Cabin'].notna().astype(int)

print("Created CabinKnown indicator.")
print("1 = Cabin recorded; 0 = Cabin missing.")



2. CABIN INVESTIGATION
Missing Cabin values: 687
Missing Cabin percentage: 77.1 %
Created CabinKnown indicator.
1 = Cabin recorded; 0 = Cabin missing.


4.Remove Duplicate Rows

In [16]:
duplicates_before = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
duplicates_after = df_clean.duplicated().sum()

print("\n3. DUPLICATE REMOVAL")
print("Duplicates before:", duplicates_before)
print("Duplicates after:", duplicates_after)
print("Rows after duplicate removal:", len(df_clean))




3. DUPLICATE REMOVAL
Duplicates before: 0
Duplicates after: 0
Rows after duplicate removal: 891


5.Standardize Categorical Values

In [17]:
for col in ['Sex', 'Embarked']:
    df_clean[col] = df_clean[col].astype('string').str.strip()

df_clean['Sex'] = df_clean['Sex'].str.lower().replace({
    'male': 'Male',
    'female': 'Female'
})

df_clean['Embarked'] = df_clean['Embarked'].str.upper()

print("\n4. CATEGORICAL STANDARDIZATION COMPLETED")




4. CATEGORICAL STANDARDIZATION COMPLETED


6.Correct Data types

In [18]:
df_clean['PassengerId'] = df_clean['PassengerId'].astype('int64')
df_clean['Survived'] = df_clean['Survived'].astype('int64')
df_clean['Pclass'] = df_clean['Pclass'].astype('int64')
df_clean['Age'] = pd.to_numeric(df_clean['Age'], errors='coerce')
df_clean['SibSp'] = df_clean['SibSp'].astype('int64')
df_clean['Parch'] = df_clean['Parch'].astype('int64')
df_clean['Fare'] = pd.to_numeric(df_clean['Fare'], errors='coerce')

print("\n5. DATA TYPES")
print(df_clean.dtypes)




5. DATA TYPES
PassengerId             int64
Survived                int64
Pclass                  int64
Name                   object
Sex            string[python]
Age                   float64
SibSp                   int64
Parch                   int64
Ticket                 object
Fare                  float64
Cabin                  object
Embarked       string[python]
CabinKnown              int64
dtype: object


7.Check invalid values

In [19]:
print("\n6. INVALID VALUE CHECK")

print("Invalid Survived values:",
      (~df_clean['Survived'].isin([0, 1])).sum())

print("Invalid Pclass values:",
      (~df_clean['Pclass'].isin([1, 2, 3])).sum())

print("Negative Age values:",
      (df_clean['Age'] < 0).sum())

print("Negative Fare values:",
      (df_clean['Fare'] < 0).sum())


6. INVALID VALUE CHECK
Invalid Survived values: 0
Invalid Pclass values: 0
Negative Age values: 0
Negative Fare values: 0


8. Detect outliers using IQR


In [20]:
print("\n7. OUTLIER DETECTION USING IQR")

outlier_report = []

for col in ['Age', 'Fare', 'SibSp', 'Parch']:

    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        (df_clean[col] < lower_bound) |
        (df_clean[col] > upper_bound)
    ).sum()

    outlier_report.append({
        'Column': col,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower Bound': lower_bound,
        'Upper Bound': upper_bound,
        'Outlier Count': int(outlier_count)
    })

outlier_df = pd.DataFrame(outlier_report)
display(outlier_df.round(2))

print(
    "\nNote: IQR flags unusual observations; "
    "it does not prove that they are errors."
)
print(
    "Outliers are retained for now because extreme passenger "
    "values may be genuine."
)









7. OUTLIER DETECTION USING IQR


,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,Age,22.00,35.0,13.00,2.50,54.50,66
1,Fare,7.91,31.0,23.09,-26.72,65.63,116
2,SibSp,0.00,1.0,1.00,-1.50,2.50,46
3,Parch,0.00,0.0,0.00,0.00,0.00,213



Note: IQR flags unusual observations; it does not prove that they are errors.
Outliers are retained for now because extreme passenger values may be genuine.


9. Missing values after cleaning

In [21]:
print("\n8. MISSING VALUES AFTER CLEANING")

missing_after = pd.DataFrame({
    'Missing Values': df_clean.isnull().sum(),
    'Missing Percentage': df_clean.isnull().mean() * 100
})

display(missing_after.round(2))




8. MISSING VALUES AFTER CLEANING


,Missing Values,Missing Percentage
PassengerId,0,0.0
Survived,0,0.0
Pclass,0,0.0
Name,0,0.0
Sex,0,0.0
Age,0,0.0
SibSp,0,0.0
Parch,0,0.0
Ticket,0,0.0
Fare,0,0.0


10. Before vs After comparison


In [22]:
before_df = pd.read_csv('/content/train.csv')

comparison = pd.DataFrame({
    'Metric': [
        'Rows',
        'Columns',
        'Duplicate Rows',
        'Missing Age Values',
        'Missing Embarked Values',
        'Missing Cabin Values'
    ],
    'Before Cleaning': [
        len(before_df),
        len(before_df.columns),
        before_df.duplicated().sum(),
        before_df['Age'].isnull().sum(),
        before_df['Embarked'].isnull().sum(),
        before_df['Cabin'].isnull().sum()
    ],
    'After Cleaning': [
        len(df_clean),
        len(df_clean.columns),
        df_clean.duplicated().sum(),
        df_clean['Age'].isnull().sum(),
        df_clean['Embarked'].isnull().sum(),
        df_clean['Cabin'].isnull().sum()
    ]
})

print("\n9. BEFORE VS AFTER COMPARISON")



9. BEFORE VS AFTER COMPARISON


11. Save cleaned dataset

In [23]:
output_file = '/content/cleaned_titanic.csv'

df_clean.to_csv(output_file, index=False)

print("\n10. CLEANED DATASET SAVED")
print("File:", output_file)
print("Final dataset shape:", df_clean.shape)



10. CLEANED DATASET SAVED
File: /content/cleaned_titanic.csv
Final dataset shape: (891, 13)


12. Final inspection


In [24]:
print("\n11. FINAL DATASET PREVIEW")
display(df_clean.head())

print("\nTASK 3 DATA CLEANING PIPELINE COMPLETED!")


11. FINAL DATASET PREVIEW


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,CabinKnown
0,1,0,3,"Braund, Mr. Owen Harris",Male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,1,3,"Heikkinen, Miss. Laina",Female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Female,35.0,1,0,113803,53.1000,C123,S,1
4,5,0,3,"Allen, Mr. William Henry",Male,35.0,0,0,373450,8.0500,NaN,S,0



TASK 3 DATA CLEANING PIPELINE COMPLETED!


In [25]:
from google.colab import files

files.download('/content/cleaned_titanic.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>